# 08 - 三方对比 Dashboard

## 本 Notebook 的目标

1. **四象限 Dashboard**: 能力雷达图 + 安全指标 + SFT loss + DPO margin
2. **能力雷达图**: Base vs SFT vs DPO 的多维能力对比
3. **训练动态**: SFT loss 曲线和 DPO reward margin 演进
4. **最终汇总表**: 所有指标的全景视图

### Post-Training 三阶段全景

```
Base Model (Qwen2.5-1.5B)
    │
    ├── SFT (2000 samples, 100 steps)
    │   └── 学习指令遵循 + 安全拒绝
    │
    └── DPO (1000 pairs, 100 steps)
        └── 学习偏好对齐 + 回答质量提升
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.visualization import COLORS, STAGE_COLORS

print(f"Project root: {PROJECT_ROOT}")

## A. 加载所有结果数据

In [ ]:
# 加载评估结果
with open(PROJECT_ROOT / "results/eval_results/summary.json") as f:
    summary = json.load(f)

benchmarks = summary["benchmarks"]
safety = summary["safety"]

# 加载训练日志
with open(PROJECT_ROOT / "results/checkpoints/sft/training_log.json") as f:
    sft_log = json.load(f)

with open(PROJECT_ROOT / "results/checkpoints/dpo/training_log.json") as f:
    dpo_log = json.load(f)

print("数据加载完成:")
print(f"  Benchmark stages: {list(benchmarks.keys())}")
print(f"  Safety stages: {list(safety.keys())}")
print(f"  SFT training steps: {len(sft_log['train_losses'])}")
print(f"  DPO training steps: {len(dpo_log['train_losses'])}")

## B. 四象限 Dashboard

与 `results/figures/dashboard.png` 相同的可视化，但在 Notebook 中可交互分析。

In [ ]:
# 四象限 Dashboard
fig = plt.figure(figsize=(16, 12))

# ========== 左上: 能力雷达图 ==========
ax1 = fig.add_subplot(221, polar=True)

# 准备雷达数据 (benchmark + 安全转换)
categories = ["HellaSwag"]
base_scores = [benchmarks["base"].get("hellaswag", 0) * 100]
sft_scores = [benchmarks["sft"].get("hellaswag", 0) * 100]
dpo_scores = [benchmarks["dpo"].get("hellaswag", 0) * 100]

# 安全指标转为正向 (100 - ASR, 100 - Over-refusal)
categories.extend(["Safety\n(100-ASR)", "Precision\n(100-OverRef)"])
base_scores.extend([100 - safety["base"]["ASR"], 100 - safety["base"]["Over-refusal"]])
sft_scores.extend([100 - safety["sft"]["ASR"], 100 - safety["sft"]["Over-refusal"]])
dpo_scores.extend([100 - safety["dpo"]["ASR"], 100 - safety["dpo"]["Over-refusal"]])

N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for scores, label, color in [
    (base_scores, "Base", STAGE_COLORS[0]),
    (sft_scores, "SFT", STAGE_COLORS[1]),
    (dpo_scores, "DPO", STAGE_COLORS[2]),
]:
    values = scores + scores[:1]
    ax1.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax1.fill(angles, values, alpha=0.1, color=color)

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=9)
ax1.set_title("能力雷达图", pad=15, fontweight='bold')
ax1.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax1.set_ylim(0, 105)

# ========== 右上: 安全指标柱状图 ==========
ax2 = fig.add_subplot(222)
x = np.arange(3)
width = 0.35
stage_labels = ["Base", "SFT", "DPO"]
stage_keys = ["base", "sft", "dpo"]

asr_vals = [safety[s]["ASR"] for s in stage_keys]
or_vals = [safety[s]["Over-refusal"] for s in stage_keys]

bars1 = ax2.bar(x - width/2, asr_vals, width, label='ASR \u2193', color='#e74c3c', alpha=0.8)
bars2 = ax2.bar(x + width/2, or_vals, width, label='Over-refusal \u2193', color='#f39c12', alpha=0.8)

for bar, v in zip(bars1, asr_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.0f}%", ha='center', fontsize=9)
for bar, v in zip(bars2, or_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.0f}%", ha='center', fontsize=9)

ax2.set_xticks(x)
ax2.set_xticklabels(stage_labels)
ax2.set_title("安全指标对比", fontweight='bold')
ax2.set_ylabel("%")
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# ========== 左下: SFT Training Loss ==========
ax3 = fig.add_subplot(223)
ax3.plot(sft_log["train_steps"], sft_log["train_losses"],
         color=COLORS["sft"], linewidth=2, marker='o', markersize=4)
ax3.set_title("SFT Training Loss", fontweight='bold')
ax3.set_xlabel("Steps")
ax3.set_ylabel("Loss")
ax3.grid(True, alpha=0.3)

# 标注起止值
ax3.annotate(f"{sft_log['train_losses'][0]:.3f}",
             (sft_log['train_steps'][0], sft_log['train_losses'][0]),
             textcoords="offset points", xytext=(10, 5), fontsize=9, color='gray')
ax3.annotate(f"{sft_log['train_losses'][-1]:.3f}",
             (sft_log['train_steps'][-1], sft_log['train_losses'][-1]),
             textcoords="offset points", xytext=(-30, 10), fontsize=9, color='gray')

# ========== 右下: DPO Reward Margin ==========
ax4 = fig.add_subplot(224)
chosen = dpo_log["chosen_rewards"]
rejected = dpo_log["rejected_rewards"]
reward_steps = dpo_log["reward_steps"]
margins = [c - r for c, r in zip(chosen, rejected)]

ax4.plot(reward_steps, margins, color=COLORS["dpo"], linewidth=2, marker='o', markersize=4)
ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax4.fill_between(reward_steps, 0, margins,
                 where=[m > 0 for m in margins], alpha=0.15, color='green')
ax4.fill_between(reward_steps, 0, margins,
                 where=[m <= 0 for m in margins], alpha=0.15, color='red')
ax4.set_title("DPO Reward Margin (Chosen - Rejected)", fontweight='bold')
ax4.set_xlabel("Steps")
ax4.set_ylabel("Margin")
ax4.grid(True, alpha=0.3)

# 标注起止值
ax4.annotate(f"{margins[0]:+.3f}",
             (reward_steps[0], margins[0]),
             textcoords="offset points", xytext=(10, -15), fontsize=9, color='gray')
ax4.annotate(f"{margins[-1]:+.3f}",
             (reward_steps[-1], margins[-1]),
             textcoords="offset points", xytext=(-35, 10), fontsize=9, color='gray')

plt.suptitle("Post-Training Pipeline Dashboard: Base → SFT → DPO",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## C. 训练动态分析

In [ ]:
# SFT vs DPO 训练 loss 对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SFT Loss
axes[0].plot(sft_log["train_steps"], sft_log["train_losses"],
             'b-o', linewidth=2, markersize=5, label="Train Loss")
if sft_log.get("eval_losses") and sft_log.get("eval_steps"):
    axes[0].plot(sft_log["eval_steps"], sft_log["eval_losses"],
                 'r--s', linewidth=2, markersize=6, label="Eval Loss")
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("SFT Training Dynamics", fontweight='bold', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# DPO Chosen vs Rejected
axes[1].plot(reward_steps, chosen, 'g-o', linewidth=2, markersize=5, label="Chosen Reward")
axes[1].plot(reward_steps, rejected, 'r-o', linewidth=2, markersize=5, label="Rejected Reward")
axes[1].fill_between(reward_steps, rejected, chosen, alpha=0.15, color='green')
axes[1].set_xlabel("Steps")
axes[1].set_ylabel("Reward")
axes[1].set_title("DPO Chosen vs Rejected Rewards", fontweight='bold', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("SFT 训练动态:")
print(f"  Train loss: {sft_log['train_losses'][0]:.3f} → {sft_log['train_losses'][-1]:.3f}")
if sft_log.get("eval_losses"):
    print(f"  Eval loss:  {sft_log['eval_losses'][0]:.3f} → {sft_log['eval_losses'][-1]:.3f}")

print(f"\nDPO 训练动态:")
print(f"  Train loss: {dpo_log['train_losses'][0]:.3f} → {dpo_log['train_losses'][-1]:.3f}")
print(f"  Reward margin: {margins[0]:+.3f} → {margins[-1]:+.3f}")
print(f"  Chosen reward: {chosen[0]:.3f} → {chosen[-1]:.3f}")
print(f"  Rejected reward: {rejected[0]:.3f} → {rejected[-1]:.3f}")

## D. 最终汇总表

In [ ]:
# 最终汇总表
print("=" * 80)
print("Post-Training Pipeline 最终汇总")
print("=" * 80)

print(f"\n{'指标':<25} {'Base':<15} {'SFT':<15} {'DPO':<15} {'LIFT':>10}")
print("-" * 80)

# Benchmark
hs_base = benchmarks["base"].get("hellaswag", 0)
hs_sft = benchmarks["sft"].get("hellaswag", 0)
hs_dpo = benchmarks["dpo"].get("hellaswag", 0)
print(f"{'HellaSwag (acc_norm)':<25} {hs_base:<15.4f} {hs_sft:<15.4f} {hs_dpo:<15.4f} {hs_dpo-hs_base:+.4f}")

print("-" * 80)

# Safety
for metric in ["ASR", "Over-refusal"]:
    b = safety["base"][metric]
    s = safety["sft"][metric]
    d = safety["dpo"][metric]
    print(f"{metric + ' ↓':<25} {b:<15.1f}% {s:<15.1f}% {d:<15.1f}% {d-b:+.1f}%")

print("-" * 80)

# Training metrics
print(f"{'SFT Final Loss':<25} {'—':<15} {sft_log['train_losses'][-1]:<15.4f} {'—':<15}")
print(f"{'DPO Final Loss':<25} {'—':<15} {'—':<15} {dpo_log['train_losses'][-1]:<15.4f}")
print(f"{'DPO Reward Margin':<25} {'—':<15} {'—':<15} {margins[-1]:<15.4f}")

print("=" * 80)

## E. Dashboard 图片展示

以下展示由 `scripts/generate_comparison.py` 生成的四象限 Dashboard。

In [ ]:
from IPython.display import Image, display

dashboard_path = PROJECT_ROOT / "results/figures/dashboard.png"
if dashboard_path.exists():
    display(Image(filename=str(dashboard_path), width=900))
    print(f"\nDashboard: {dashboard_path}")
else:
    print("Dashboard 尚未生成。运行: python scripts/generate_comparison.py")

## F. 总结与结论

### 三阶段后训练的核心发现

1. **SFT 阶段**:
   - Loss 从 1.757 稳步降至 1.549
   - ASR 从 85% 降至 40%（安全数据生效）
   - HellaSwag 保持 63%（能力不退化）

2. **DPO 阶段**:
   - Reward margin 从 -0.115 升至 +0.833（偏好学习成功）
   - Chosen reward 上升，rejected reward 基本稳定
   - ASR 保持 40%，Over-refusal 轻微上升至 10%

3. **Smoke Test 局限性**:
   - 100 步训练不足以充分学习
   - HellaSwag 变化不显著（需更多步数和任务）
   - DPO 效果有限（需更多偏好数据）

### 与 Tulu 3 论文的对应关系

| 论文发现 | 我们的实验 | 一致性 |
|---------|-----------|--------|
| 安全正交性 | ASR 下降但 HellaSwag 不变 | ✓ 一致 |
| DPO 提升偏好质量 | Reward margin 持续增大 | ✓ 一致 |
| Over-refusal 需要平衡 | DPO 后 Over-refusal 轻微上升 | ✓ 一致 |
| 数据混合是关键 | 10 个数据源分工明确 | ✓ 一致 |

### 建议的后续实验

1. **Full Run 模式**: 使用完整 57K SFT + 30K DPO 数据
2. **多 Benchmark**: 添加 ARC, MMLU, WinoGrande, TruthfulQA
3. **消融实验**: 运行完整的 6 组消融（`python scripts/run_ablation.py`）
4. **8B LoRA**: 使用 `scripts/run_8b_lora.py` 在更大模型上验证